In [1]:
import os
import anndata
import pandas as pd
import numpy as np

folder_path = "/exports/archive/hg-exon-skip/alireza/expression_matrices/WMB-10Xv2/20230630/"

metadata_path = "/exports/archive/hg-exon-skip/alireza/expression_matrices/metadata/WMB-10X/20230830/views/cell_metadata_with_cluster_annotation.csv"
metadata = pd.read_csv(metadata_path , index_col=0)
metadata = metadata[(metadata['library_method'] == '10Xv2')]

concatenated_data = None

# Loop through files in the folder
for file_name in os.listdir(folder_path):
    if file_name.endswith("raw.h5ad"):
        # Form the full path of the current file
        file_path = os.path.join(folder_path, file_name)

        # Read the current h5ad file
        adata = anndata.read_h5ad(file_path)
        gene_symbols = adata.var['gene_symbol'].tolist()
        adata.var_names = gene_symbols
        adata.var_names_make_unique()
        subsampled_df = metadata.groupby('class', group_keys=False).apply(lambda group: group.sample(frac=0.1, random_state=42))
        adata.obs['cluster_ann'] = adata.obs.index.isin(subsampled_df.index)
        adata_subset = adata[adata.obs.cluster_ann]
        # Concatenate data
        if concatenated_data is None:
            concatenated_data = adata_subset
        else:
            concatenated_data = anndata.concat([concatenated_data, adata_subset], index_unique=None)

# Save the concatenated data to a new h5ad file
concatenated_data.write("/exports/archive/hg-exon-skip/alireza/expression_matrices/file_sampled.h5ad")

/tmp/ipykernel_641957/640779834.py:9: DtypeWarning: Columns (16) have mixed types. Specify dtype option on import or set low_memory=False.
  metadata = pd.read_csv(metadata_path , index_col=0)
